In [28]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

### Q1 Spark Version

In [ ]:
print(spark.version)

In [7]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [10]:
df = df.repartition(4)
df.write.mode("overwrite").parquet('yellow/2025/11/', )

### Q3 Count records

In [13]:
df.withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .filter("pickup_date = '2025-11-15'") \
    .count()

162604

### Q4 Longest trip

In [34]:
df.withColumn('trip_duration', 
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime'))/60/60
).select(
    F.round(F.max('trip_duration'), 2).alias('max_duration')
).show()

[Stage 57:=============================>                            (4 + 4) / 8]

+------------+
|max_duration|
+------------+
|       90.65|
+------------+



In [41]:
# Try another approach with udf:

def trip_duration_fn(dropoff, pickup):
    if dropoff is None or pickup is None:
        return None
        
    delta = dropoff - pickup 
    
    return float(delta.total_seconds())

trip_duration_udf = F.udf(trip_duration_fn, returnType=types.FloatType())

df.withColumn('trip_duration', 
    trip_duration_udf(F.col('tpep_dropoff_datetime'), F.col('tpep_pickup_datetime'))
).select(
    F.round(F.max('trip_duration')/60/60, 2).alias('max_duration')
).show()

[Stage 83:>                                                         (0 + 4) / 4]

+------------+
|max_duration|
+------------+
|       90.65|
+------------+



### Q6 Least Frequent Pickup Location Zone

In [42]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [45]:
df_join = df.join(df_zones, df.PULocationID == df_zones.LocationID)

In [47]:
df_join.withColumn('Zone',df_join.Zone).groupby('Zone').count().show()

TypeError: DataFrame.withColumn() missing 1 required positional argument: 'col'